# YB Writer — train YouTube Buddy's own free writing model

This notebook makes a small writing model for youtubebuddy.com's writing tools (scripts, hooks, titles, descriptions,
story outlines, dialogue, shot lists, prompts, social posts, rewrites, title A/B, SEO and content calendars) in
English, Hindi, Telugu, Tamil, Kannada, Malayalam, Bengali and Marathi.

1. A big free model (**Qwen3-14B**, Apache-2.0) writes thousands of answers to the site's exact prompts. Every answer
   is checked; bad ones (wrong language, wrong JSON, fake SEO scores) are thrown away.
2. A small model (**Qwen3-1.7B**, Apache-2.0) learns from the good answers.
3. It is tested on examples it never saw, before and after training.
4. It is saved as one file, `yb-writer-q4_k_m.gguf` (~1.1 GB), which runs on the ASUS processor for ₹0.

**Before you run:** in the right-hand panel set **Accelerator → GPU T4 x2** and switch **Internet → On**
(both need a phone-verified Kaggle account).

**To run:** click **Save Version → Save & Run All (Commit)**. It keeps running when you close the browser.
It takes about 5–8 hours of your weekly free GPU time. When it finishes, open the version → **Output** tab and
download `yb-writer-q4_k_m.gguf` and `report.md`.


In [ ]:
# Settings. QUICK = True does a ~1-hour trial with 10% of the data (to check everything works first).
QUICK = False
SCALE = 0.1 if QUICK else 1.0
import os, subprocess
os.environ["TEACHER_VLLM"] = "Qwen/Qwen3-14B-AWQ"   # big teacher (2x T4)
os.environ["TEACHER_HF"] = "Qwen/Qwen3-8B"          # fallback teacher if vLLM can't start
os.environ["STUDENT"] = "Qwen/Qwen3-1.7B"           # the small model the website will run
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


In [ ]:
# Write the training scripts (exported from the YouTube Buddy repo, brain/kaggle/writer/)
import os
os.makedirs('/kaggle/working/ybw', exist_ok=True)
FILES = {"prompts.json": "{\n \"tasks\": {\n  \"script\": [\n   \"You are a senior YouTube scriptwriter for Indian creators.\",\n   \"Write a {format} script. Brief: {brief}\\nTone: {tone}. Audience: {audience}. Length: {length}.\",\n   \"{\\\"title\\\": str, \\\"logline\\\": str, \\\"scenes\\\": [{\\\"heading\\\": str, \\\"narration\\\": str, \\\"visuals\\\": str, \\\"on_screen_text\\\": str, \\\"duration_s\\\": number}], \\\"cta\\\": str}\"\n  ],\n  \"hooks\": [\n   \"You write scroll-stopping video hooks.\",\n   \"Write {count} different opening hooks (first 3 seconds). Brief: {brief}. Tone: {tone}. Audience: {audience}.\",\n   \"{\\\"items\\\": [{\\\"text\\\": str, \\\"technique\\\": str, \\\"score\\\": number(1-10), \\\"why\\\": str}]}\"\n  ],\n  \"titles\": [\n   \"You are a YouTube packaging expert.\",\n   \"Write {count} titles under 70 characters. Brief: {brief}. Tone: {tone}. Audience: {audience}.\",\n   \"{\\\"items\\\": [{\\\"text\\\": str, \\\"curiosity\\\": number(1-10), \\\"clarity\\\": number(1-10), \\\"chars\\\": number, \\\"why\\\": str}]}\"\n  ],\n  \"description\": [\n   \"You write YouTube descriptions that rank.\",\n   \"Write an SEO description and tags. Brief: {brief}. Links to include: {links}.\",\n   \"{\\\"description\\\": str, \\\"tags\\\": [str], \\\"hashtags\\\": [str], \\\"keywords\\\": [str]}\"\n  ],\n  \"outline\": [\n   \"You are a story editor.\",\n   \"Outline this as a {structure} story. Brief: {brief}. Tone: {tone}. Audience: {audience}.\",\n   \"{\\\"title\\\": str, \\\"beats\\\": [{\\\"name\\\": str, \\\"summary\\\": str, \\\"purpose\\\": str, \\\"minutes\\\": number}]}\"\n  ],\n  \"dialogue\": [\n   \"You write natural, funny, culturally specific dialogue.\",\n   \"Write a {length} scene. Brief: {brief}. Characters: {characters}. Tone: {tone}.\",\n   \"{\\\"title\\\": str, \\\"setting\\\": str, \\\"lines\\\": [{\\\"character\\\": str, \\\"line\\\": str, \\\"direction\\\": str}]}\"\n  ],\n  \"shotlist\": [\n   \"You are a director of photography.\",\n   \"Break this script into shots for a {aspect_ratio} video:\\n{script}\",\n   \"{\\\"shots\\\": [{\\\"prompt\\\": str, \\\"camera_move\\\": one of [static, push in, pull out, pan left, pan right, tilt up, tilt down, orbit, crane up, tracking, handheld], \\\"camera_angle\\\": str, \\\"lens\\\": str, \\\"duration\\\": number(4-10), \\\"narration\\\": str}]}\"\n  ],\n  \"prompt\": [\n   \"You write prompts for state-of-the-art {target} generation models (Veo, Kling, FLUX).\",\n   \"Turn this idea into 3 precise prompts in {style} style. Include subject, action, setting, camera, lens, light, mood. Idea: {prompt}\",\n   \"{\\\"items\\\": [{\\\"text\\\": str, \\\"why\\\": str}]}\"\n  ],\n  \"social\": [\n   \"You are a social media copywriter.\",\n   \"Write posts for {platforms}. Brief: {brief}. Tone: {tone}. Audience: {audience}.\",\n   \"{\\\"items\\\": [{\\\"platform\\\": str, \\\"text\\\": str, \\\"hashtags\\\": [str]}]}\"\n  ],\n  \"rewrite\": [\n   \"You are an editor.\",\n   \"Rewrite to be {goal}, tone {tone}:\\n{text}\",\n   \"{\\\"text\\\": str, \\\"changes\\\": [str]}\"\n  ],\n  \"title_ab\": [\n   \"You are a YouTube packaging analyst. Be honest; these are judgments, not predictions of real click-through rate.\",\n   \"Compare these titles for audience {audience}:\\n{titles}\",\n   \"{\\\"items\\\": [{\\\"text\\\": str, \\\"clarity\\\": number(1-10), \\\"curiosity\\\": number(1-10), \\\"specificity\\\": number(1-10), \\\"chars\\\": number, \\\"verdict\\\": str}], \\\"winner\\\": str, \\\"why\\\": str}\"\n  ],\n  \"seo\": [\n   \"You are a YouTube SEO analyst.\",\n   \"Grade this metadata for keyword '{keyword}'. Title: {title}\\nDescription: {description}\\nTags: {tags}\",\n   \"{\\\"score\\\": number(0-100), \\\"checks\\\": [{\\\"name\\\": str, \\\"pass\\\": bool, \\\"detail\\\": str}], \\\"improved_title\\\": str, \\\"improved_tags\\\": [str]}\"\n  ],\n  \"calendar\": [\n   \"You are a content strategist.\",\n   \"Plan 4 weeks, {per_week} uploads per week, for niche {niche}, audience {audience}.\",\n   \"{\\\"weeks\\\": [{\\\"week\\\": number, \\\"items\\\": [{\\\"day\\\": str, \\\"format\\\": str, \\\"title\\\": str, \\\"hook\\\": str}]}]}\"\n  ]\n },\n \"lang_names\": {\n  \"en\": \"English\",\n  \"hi\": \"Hindi, in Devanagari script\",\n  \"te\": \"Telugu, in Telugu script\",\n  \"ta\": \"Tamil, in Tamil script\",\n  \"kn\": \"Kannada, in Kannada script\",\n  \"ml\": \"Malayalam, in Malayalam script\",\n  \"bn\": \"Bengali, in Bengali script\",\n  \"mr\": \"Marathi, in Devanagari script\",\n  \"gu\": \"Gujarati, in Gujarati script\",\n  \"pa\": \"Punjabi, in Gurmukhi script\",\n  \"ur\": \"Urdu, in Urdu script\"\n }\n}", "ybw_data.py": "\"\"\"YB Writer: builds training examples for the site's writing tools and checks answers.\n\nThe prompts come from prompts.json, exported from backend/app/tools/engines.py (TASKS + LANG_NAMES), so the\ntrained model sees exactly what the website sends it. Re-export prompts.json whenever those prompts change.\n\"\"\"\nimport json\nimport random\nimport re\n\nALL_LANGS = [\"en\", \"hi\", \"te\", \"ta\", \"kn\", \"ml\", \"bn\", \"mr\"]\nLANG_WEIGHTS = [40, 20, 15, 7, 4.5, 4.5, 4.5, 4.5]\nTONES = [\"friendly\", \"energetic\", \"calm\", \"funny\", \"serious\", \"inspiring\", \"dramatic\"]\nLENGTHS = [\"short\", \"medium\", \"long\"]\nFORMATS = [\"YouTube video\", \"Short / Reel\", \"Ad\", \"Explainer\", \"Podcast\"]\nSTRUCTURES = [\"three-act\", \"hero's journey\", \"problem-solution\", \"listicle\", \"before-after\"]\nGOALS = [\"clearer\", \"shorter\", \"longer\", \"simpler\", \"more persuasive\", \"Hinglish\"]\nPLATFORMS = [\"Instagram\", \"YouTube Shorts\", \"X\", \"LinkedIn\", \"Facebook\"]\nASPECTS = [\"16:9\", \"9:16\", \"1:1\"]\nSTYLES = [\"cinematic photo\", \"editorial\", \"anime\", \"3D render\", \"watercolor\", \"oil painting\", \"comic\", \"pixel art\", \"low poly\",\n          \"line art\", \"claymation\", \"vaporwave\", \"Indian miniature\", \"Madhubani\", \"Bollywood poster\", \"documentary\"]\nCAMERA_MOVES = {\"static\", \"push in\", \"pull out\", \"pan left\", \"pan right\", \"tilt up\", \"tilt down\", \"orbit\", \"crane up\", \"tracking\", \"handheld\"}\n\nNICHES = [\n    \"Hyderabad street food\", \"Telugu cooking at home\", \"budget travel in India\", \"Kerala backwaters travel\", \"Himalayan treks\",\n    \"personal finance for salaried Indians\", \"stock market basics\", \"mutual funds for beginners\", \"UPSC preparation\",\n    \"JEE physics tips\", \"class 10 maths\", \"spoken English for beginners\", \"coding for beginners in Python\", \"AI tools for students\",\n    \"smartphone reviews under 20000 rupees\", \"laptop buying guide\", \"BGMI gaming\", \"Free Fire gaming\", \"chess for beginners\",\n    \"cricket analysis\", \"kabaddi highlights\", \"home workouts without equipment\", \"yoga for back pain\", \"healthy Indian breakfast\",\n    \"skincare for Indian skin\", \"budget fashion for college students\", \"saree draping styles\", \"wedding vlogs\", \"family vlogs\",\n    \"village life vlogs\", \"farming and organic vegetables\", \"small business ideas in India\", \"freelancing from home\",\n    \"Instagram growth tips\", \"YouTube growth for small channels\", \"Telugu movie reviews\", \"Bollywood music covers\",\n    \"carnatic music lessons\", \"devotional temple tours\", \"Indian history stories\", \"science experiments for kids\",\n    \"pet care for dogs\", \"car reviews for Indian roads\", \"bike rides and road trips\", \"real estate tips for first homes\",\n    \"mobile photography\", \"video editing tutorials\", \"comedy sketches\", \"motivational stories\", \"book summaries\",\n    \"language learning Hindi to English\", \"kids rhymes and stories\", \"festival recipes for Diwali\", \"Sankranti traditions\",\n    \"monsoon travel\", \"tech news in simple words\", \"government schemes explained\", \"career advice after 12th\",\n    \"interview preparation\", \"productivity for students\",\n]\n\nSEED_SYSTEM = \"You create realistic test inputs for a writing app used by Indian YouTube creators. Reply with ONE JSON object only.\"\nSEED_USER = \"\"\"Niche: {niche}.\nCreate varied, realistic inputs that creators in this niche would type. Mix English, Hinglish and a few in Hindi or Telugu.\nReturn JSON with exactly these keys:\n\"briefs\": 6 items {{\"brief\": 1-3 sentences describing one specific video, \"audience\": who it is for}},\n\"dialogues\": 2 items {{\"brief\": a short scene idea, \"characters\": 2-3 lines like \"Name: trait\"}},\n\"scripts\": 2 short video scripts (4-6 sentences each, plain text),\n\"texts\": 4 short informal captions or sentences a creator wants rewritten,\n\"title_sets\": 2 lists of 3-4 candidate titles of mixed quality,\n\"metadata\": 3 items {{\"title\", \"description\", \"tags\", \"keyword\", \"quality\"}} where quality is \"weak\" for the first (vague title, tiny description, few tags), \"average\" for the second and \"strong\" for the third,\n\"ideas\": 3 short visual ideas for an image or video prompt.\"\"\"\n\n\ndef lang_hint(p, lang_names):\n    code = (p.get(\"language\") or \"en\").strip()\n    return f\" Write in {lang_names.get(code, code)} (language code {code}). Keep JSON keys in English.\"\n\n\nclass _Safe(dict):\n    def __missing__(self, k):\n        return \"\"\n\n\ndef build_prompt(prompts, task, params):\n    \"\"\"Exactly what backend run_llm() sends: (system, user).\"\"\"\n    sys_t, user_t, schema = prompts[\"tasks\"][task]\n    params = public_params(params)\n    vals = _Safe({k: (\", \".join(v) if isinstance(v, list) else v) for k, v in params.items()})\n    system = sys_t.format_map(vals) + lang_hint(params, prompts[\"lang_names\"]) + \"\\nReply with ONE JSON object only, matching: \" + schema\n    return system, user_t.format_map(vals)\n\n\ndef parse_json(text):\n    \"\"\"Same rules as backend providers.parse_json; returns None instead of raising.\"\"\"\n    text = re.sub(r\"<think>.*?</think>\", \"\", text or \"\", flags=re.S).strip()\n    m = re.search(r\"```(?:json)?\\s*(\\{.*\\})\\s*```\", text, re.S)\n    if m:\n        text = m.group(1)\n    else:\n        a, b = text.find(\"{\"), text.rfind(\"}\")\n        if a >= 0 and b > a:\n            text = text[a:b + 1]\n    try:\n        out = json.loads(text)\n    except json.JSONDecodeError:\n        return None\n    return out if isinstance(out, dict) else None\n\n\ndef pick_lang(rng):\n    return rng.choices(ALL_LANGS, LANG_WEIGHTS)[0]\n\n\ndef make_params(task, seed, rng):\n    \"\"\"Random but realistic tool inputs for one task, from one niche's seed object.\"\"\"\n    b = rng.choice(seed[\"briefs\"])\n    common = {\"brief\": b[\"brief\"], \"audience\": b.get(\"audience\", \"\"), \"tone\": rng.choice(TONES)}\n    if task == \"script\":\n        return {**common, \"language\": pick_lang(rng), \"length\": rng.choice([\"short\", \"short\", \"medium\"]), \"format\": rng.choice(FORMATS)}\n    if task == \"hooks\":\n        return {**common, \"language\": pick_lang(rng), \"count\": rng.randint(3, 8)}\n    if task == \"titles\":\n        return {**common, \"language\": pick_lang(rng), \"count\": rng.randint(4, 10)}\n    if task == \"description\":\n        links = rng.choice([\"\", \"Instagram: @\" + re.sub(r\"\\W\", \"\", b[\"brief\"].split()[0].lower())[:12] + \"_official\",\n                            \"Shop: https://example.in/store\", \"Telegram: t.me/mychannel\"])\n        return {\"brief\": b[\"brief\"], \"language\": pick_lang(rng), \"links\": links}\n    if task == \"outline\":\n        return {**common, \"language\": pick_lang(rng), \"structure\": rng.choice(STRUCTURES)}\n    if task == \"dialogue\":\n        d = rng.choice(seed[\"dialogues\"])\n        return {\"brief\": d[\"brief\"], \"characters\": d[\"characters\"], \"tone\": rng.choice(TONES), \"audience\": b.get(\"audience\", \"\"),\n                \"language\": pick_lang(rng), \"length\": rng.choice([\"short\", \"medium\"])}\n    if task == \"shotlist\":\n        return {\"script\": rng.choice(seed[\"scripts\"]), \"aspect_ratio\": rng.choice(ASPECTS)}\n    if task == \"prompt\":\n        return {\"prompt\": rng.choice(seed[\"ideas\"]), \"target\": rng.choice([\"video\", \"image\"]), \"style\": rng.choice(STYLES)}\n    if task == \"social\":\n        return {**common, \"language\": pick_lang(rng), \"platforms\": rng.sample(PLATFORMS, rng.randint(1, 3))}\n    if task == \"rewrite\":\n        return {\"text\": rng.choice(seed[\"texts\"]), \"tone\": rng.choice(TONES), \"language\": pick_lang(rng), \"goal\": rng.choice(GOALS)}\n    if task == \"title_ab\":\n        return {\"titles\": \"\\n\".join(rng.choice(seed[\"title_sets\"])), \"audience\": b.get(\"audience\", \"\")}\n    if task == \"seo\":\n        m = rng.choice(seed[\"metadata\"])\n        tags = m[\"tags\"] if isinstance(m[\"tags\"], str) else \", \".join(m[\"tags\"])\n        return {\"title\": m[\"title\"], \"description\": m[\"description\"], \"tags\": tags, \"keyword\": m[\"keyword\"], \"_quality\": m.get(\"quality\", \"\")}\n    if task == \"calendar\":\n        return {\"niche\": seed[\"niche\"], \"per_week\": rng.randint(1, 4), \"audience\": b.get(\"audience\", \"\"), \"language\": pick_lang(rng)}\n    raise ValueError(task)\n\n\ndef seed_ok(s):\n    try:\n        return (len(s[\"briefs\"]) >= 3 and all(x[\"brief\"] for x in s[\"briefs\"]) and s[\"dialogues\"] and s[\"scripts\"] and s[\"texts\"]\n                and all(isinstance(t, list) and len(t) >= 2 for t in s[\"title_sets\"]) and len(s[\"metadata\"]) >= 3 and s[\"ideas\"])\n    except (KeyError, TypeError):\n        return False\n\n\nSCRIPT_RANGES = {\"hi\": (0x0900, 0x097F), \"mr\": (0x0900, 0x097F), \"te\": (0x0C00, 0x0C7F), \"ta\": (0x0B80, 0x0BFF),\n                 \"kn\": (0x0C80, 0x0CFF), \"ml\": (0x0D00, 0x0D7F), \"bn\": (0x0980, 0x09FF)}\n# Fields whose text should be in the chosen language (others, e.g. camera_move or tags, may stay English).\nTEXT_FIELDS = {\"script\": [\"title\", \"logline\", \"narration\", \"cta\"], \"hooks\": [\"text\"], \"titles\": [\"text\"], \"description\": [\"description\"],\n               \"outline\": [\"title\", \"summary\"], \"dialogue\": [\"line\"], \"social\": [\"text\"], \"rewrite\": [\"text\"], \"calendar\": [\"title\", \"hook\"]}\n\n\ndef _strings(o, keys):\n    if isinstance(o, dict):\n        for k, v in o.items():\n            if isinstance(v, str) and k in keys:\n                yield v\n            else:\n                yield from _strings(v, keys)\n    elif isinstance(o, list):\n        for v in o:\n            yield from _strings(v, keys)\n\n\ndef language_ok(task, lang, out):\n    \"\"\"Share of letters in the target script. Hindi output in Latin letters was the old model's main failure.\"\"\"\n    if task not in TEXT_FIELDS:\n        return True\n    text = \" \".join(_strings(out, set(TEXT_FIELDS[task])))\n    letters = [c for c in text if c.isalpha()]\n    if len(letters) < 5:\n        return False\n    if lang in SCRIPT_RANGES:\n        lo, hi = SCRIPT_RANGES[lang]\n        return sum(lo <= ord(c) <= hi for c in letters) / len(letters) >= 0.6\n    return sum(c.isascii() for c in letters) / len(letters) >= 0.9\n\n\ndef _items(out, key=\"items\", n=None):\n    v = out.get(key)\n    return isinstance(v, list) and len(v) > 0 and (n is None or len(v) == n) and all(isinstance(x, dict) for x in v)\n\n\ndef check(task, params, out):\n    \"\"\"Returns (ok, reason). Also fixes numbers the model can't count reliably (title lengths).\"\"\"\n    if out is None:\n        return False, \"not JSON\"\n    lang = params.get(\"language\") or \"en\"\n    try:\n        if task == \"script\":\n            ok = isinstance(out.get(\"title\"), str) and _items(out, \"scenes\") and all(\"narration\" in s for s in out[\"scenes\"])\n        elif task == \"hooks\":\n            ok = _items(out, n=int(params[\"count\"])) and all(x.get(\"text\") for x in out[\"items\"])\n        elif task == \"titles\":\n            ok = _items(out, n=int(params[\"count\"])) and all(x.get(\"text\") for x in out[\"items\"])\n            if ok:\n                for x in out[\"items\"]:\n                    x[\"chars\"] = len(x[\"text\"])\n                ok = all(x[\"chars\"] <= 70 for x in out[\"items\"])\n        elif task == \"description\":\n            ok = isinstance(out.get(\"description\"), str) and len(out[\"description\"]) > 80 and isinstance(out.get(\"tags\"), list)\n            if ok and params.get(\"links\"):\n                ok = params[\"links\"].split(\": \", 1)[-1] in out[\"description\"]\n        elif task == \"outline\":\n            ok = _items(out, \"beats\") and all(b.get(\"name\") and b.get(\"summary\") for b in out[\"beats\"])\n        elif task == \"dialogue\":\n            ok = _items(out, \"lines\") and len(out[\"lines\"]) >= 4 and all(x.get(\"character\") and x.get(\"line\") for x in out[\"lines\"])\n        elif task == \"shotlist\":\n            ok = _items(out, \"shots\") and all(s.get(\"prompt\") and s.get(\"camera_move\") in CAMERA_MOVES for s in out[\"shots\"])\n        elif task == \"prompt\":\n            ok = _items(out, n=3) and all(len(x.get(\"text\", \"\")) > 40 for x in out[\"items\"])\n        elif task == \"social\":\n            plats = params[\"platforms\"] if isinstance(params[\"platforms\"], list) else [params[\"platforms\"]]\n            ok = _items(out) and {x.get(\"platform\") for x in out[\"items\"]} >= set(plats) and all(x.get(\"text\") for x in out[\"items\"])\n        elif task == \"rewrite\":\n            ok = isinstance(out.get(\"text\"), str) and len(out[\"text\"]) > 3\n            if ok and params.get(\"goal\") == \"Hinglish\":   # Hindi words in Latin letters, so the script check doesn't apply\n                latin = sum(c.isascii() for c in out[\"text\"] if c.isalpha()) / max(1, sum(c.isalpha() for c in out[\"text\"])) > 0.9\n                return latin, \"\" if latin else \"Hinglish must be Latin letters\"\n        elif task == \"title_ab\":\n            n = len([t for t in params[\"titles\"].split(\"\\n\") if t.strip()])\n            ok = _items(out, n=n) and isinstance(out.get(\"winner\"), str) and out.get(\"why\")\n            if ok:\n                for x in out[\"items\"]:\n                    x[\"chars\"] = len(x.get(\"text\", \"\"))\n        elif task == \"seo\":\n            s = out.get(\"score\")\n            ok = isinstance(s, (int, float)) and 0 <= s <= 100 and _items(out, \"checks\") and out.get(\"improved_title\")\n            q = params.get(\"_quality\")\n            if ok and q == \"weak\" and s > 45:\n                return False, f\"weak metadata scored {s}\"\n            if ok and q == \"strong\" and s < 55:\n                return False, f\"strong metadata scored {s}\"\n        elif task == \"calendar\":\n            pw = int(params[\"per_week\"])\n            ok = _items(out, \"weeks\", n=4) and all(_items(w, \"items\", n=pw) and all(i.get(\"title\") for i in w[\"items\"]) for w in out[\"weeks\"])\n        else:\n            return False, \"unknown task\"\n    except (TypeError, ValueError, AttributeError, KeyError):\n        return False, \"wrong shape\"\n    if not ok:\n        return False, \"wrong shape\"\n    if task != \"rewrite\" or params.get(\"goal\") != \"Hinglish\":\n        if not language_ok(task, lang, out):\n            return False, f\"not in {lang}\"\n    return True, \"\"\n\n\ndef public_params(p):\n    return {k: v for k, v in p.items() if not k.startswith(\"_\")}\n\n\ndef jobs(prompts, seeds, counts, seed=7):\n    \"\"\"[(task, params)] for the teacher, spread across niches.\"\"\"\n    rng = random.Random(seed)\n    out = []\n    for task, n in counts.items():\n        for i in range(n):\n            out.append((task, make_params(task, seeds[i % len(seeds)], rng)))\n    rng.shuffle(out)\n    return out\n", "gen.py": "\"\"\"Step 1 on Kaggle: a big open model (the \"teacher\") writes good answers for the site's writing tools.\n\n    python gen.py --out /kaggle/working/data.jsonl            (resumes if the file exists)\n    python gen.py --fake --out /tmp/data.jsonl --scale 0.02   (offline self-test with a stub teacher)\n\nTeacher: Qwen3-14B-AWQ on vLLM (2x T4), falling back to Qwen3-8B on Hugging Face transformers. Both Apache-2.0.\nEvery answer is checked (JSON shape, counts, language script, honest SEO scores) before it is kept.\n\"\"\"\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport ybw_data as D  # noqa: E402\n\nCOUNTS = {\"script\": 320, \"hooks\": 220, \"titles\": 260, \"description\": 220, \"outline\": 170, \"dialogue\": 170, \"shotlist\": 170,\n          \"prompt\": 150, \"social\": 220, \"rewrite\": 270, \"title_ab\": 170, \"seo\": 240, \"calendar\": 170}\nMAX_ANSWER_TOKENS = 1400          # the website asks the local model for at most 1500 tokens\n\n\nclass Teacher:\n    def __init__(self, fake=False):\n        self.kind = \"fake\" if fake else None\n        if fake:\n            return\n        try:\n            from vllm import LLM, SamplingParams\n            self.llm = LLM(model=os.environ.get(\"TEACHER_VLLM\", \"Qwen/Qwen3-14B-AWQ\"), tensor_parallel_size=int(os.environ.get(\"TP\", \"2\")),\n                           dtype=\"half\", max_model_len=6144, gpu_memory_utilization=0.90, enforce_eager=False)\n            self.SP = SamplingParams\n            self.kind = \"vllm\"\n            self.ask([\"Say OK in JSON.\"], [\"{}\"], 20, 0.0)      # smoke test\n        except Exception as e:                                    # noqa: BLE001\n            print(\"vLLM unavailable, falling back to transformers:\", repr(e)[:300], flush=True)\n            import torch\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            name = os.environ.get(\"TEACHER_HF\", \"Qwen/Qwen3-8B\")\n            self.tok = AutoTokenizer.from_pretrained(name, padding_side=\"left\")\n            self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map=\"auto\")\n            self.torch = torch\n            self.kind = \"hf\"\n        print(\"teacher:\", self.kind, flush=True)\n\n    def ask(self, systems, users, max_tokens, temperature=0.7):\n        if self.kind == \"fake\":\n            return [fake_answer(s, u) for s, u in zip(systems, users)]\n        msgs = [[{\"role\": \"system\", \"content\": s}, {\"role\": \"user\", \"content\": u}] for s, u in zip(systems, users)]\n        if self.kind == \"vllm\":\n            sp = self.SP(max_tokens=max_tokens, temperature=temperature, top_p=0.9)\n            res = self.llm.chat(msgs, sp, use_tqdm=False, chat_template_kwargs={\"enable_thinking\": False})\n            return [r.outputs[0].text for r in res]\n        outs = []\n        for i in range(0, len(msgs), 8):\n            batch = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in msgs[i:i + 8]]\n            enc = self.tok(batch, return_tensors=\"pt\", padding=True).to(self.model.device)\n            with self.torch.no_grad():\n                g = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=True, temperature=temperature, top_p=0.9)\n            outs += self.tok.batch_decode(g[:, enc[\"input_ids\"].shape[1]:], skip_special_tokens=True)\n        return outs\n\n\ndef fake_answer(system, user):\n    \"\"\"Stub teacher for the offline self-test: shape-correct JSON so the plumbing can be checked without a GPU.\"\"\"\n    if user.startswith(\"Niche:\"):\n        n = user.split(\"\\n\")[0][7:-1]\n        return json.dumps({\"briefs\": [{\"brief\": f\"A video about {n} part {i}\", \"audience\": \"Indian viewers\"} for i in range(6)],\n                           \"dialogues\": [{\"brief\": \"Two friends talk\", \"characters\": \"Ravi: loud\\nMeera: calm\"}] * 2,\n                           \"scripts\": [\"Open on the street. Walk in. Taste. React. Point to the next stall.\"] * 2,\n                           \"texts\": [\"bro this is insane\", \"pls like and share\"] * 2,\n                           \"title_sets\": [[\"Best food ever\", \"I tried 5 foods for ₹500\", \"Food vlog\"]] * 2,\n                           \"metadata\": [{\"title\": \"food\", \"description\": \"vlog\", \"tags\": \"food\", \"keyword\": \"food\", \"quality\": q} for q in (\"weak\", \"average\", \"strong\")],\n                           \"ideas\": [\"street stall at night\"] * 3})\n    return '{\"text\": \"ok\"}'\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--out\", default=\"/kaggle/working/data.jsonl\")\n    ap.add_argument(\"--fake\", action=\"store_true\")\n    ap.add_argument(\"--scale\", type=float, default=1.0, help=\"multiply the example counts (0.1 for a quick try)\")\n    ap.add_argument(\"--rounds\", type=int, default=2, help=\"seed rounds per niche\")\n    a = ap.parse_args()\n    here = os.path.dirname(os.path.abspath(__file__))\n    prompts = json.load(open(os.path.join(here, \"prompts.json\")))\n    seeds_path = os.path.join(os.path.dirname(a.out), \"seeds.json\")\n    t = Teacher(a.fake)\n    t0 = time.time()\n\n    if os.path.exists(seeds_path):\n        seeds = json.load(open(seeds_path))\n    else:\n        niches = D.NICHES * a.rounds\n        if a.scale < 1:\n            niches = niches[:max(3, int(len(niches) * a.scale))]\n        outs = t.ask([D.SEED_SYSTEM] * len(niches), [D.SEED_USER.format(niche=n) for n in niches], 3000, 0.9)\n        seeds = []\n        for n, o in zip(niches, outs):\n            s = D.parse_json(o)\n            if s and D.seed_ok(s):\n                s[\"niche\"] = n\n                seeds.append(s)\n        json.dump(seeds, open(seeds_path, \"w\"), ensure_ascii=False)\n    print(f\"seeds: {len(seeds)} niches ok ({time.time() - t0:.0f}s)\", flush=True)\n    if not seeds:\n        sys.exit(\"No usable seeds: the teacher isn't answering in JSON. Check the log above.\")\n\n    counts = {k: max(1, int(v * a.scale)) for k, v in COUNTS.items()}\n    todo = D.jobs(prompts, seeds, counts)\n    done = 0\n    if os.path.exists(a.out):\n        done = sum(1 for _ in open(a.out))\n        print(\"resuming after\", done, \"jobs already tried\", flush=True)\n    stats = {}\n    with open(a.out, \"a\") as f:\n        for i in range(done, len(todo), 64):\n            chunk = todo[i:i + 64]\n            built = [D.build_prompt(prompts, task, p) for task, p in chunk]\n            outs = t.ask([s for s, _ in built], [u for _, u in built], MAX_ANSWER_TOKENS, 0.7)\n            for (task, p), (s, u), o in zip(chunk, built, outs):\n                ans = D.parse_json(o)\n                ok, why = D.check(task, p, ans)\n                st = stats.setdefault(task, [0, 0])\n                st[0] += 1\n                st[1] += ok\n                f.write(json.dumps({\"task\": task, \"lang\": p.get(\"language\", \"en\"), \"params\": p, \"system\": s, \"user\": u,\n                                    \"answer\": json.dumps(ans, ensure_ascii=False, separators=(\",\", \":\")) if ok else None,\n                                    \"reject\": why or None}, ensure_ascii=False) + \"\\n\")\n            f.flush()\n            print(f\"{min(i + 64, len(todo))}/{len(todo)} jobs, {time.time() - t0:.0f}s, kept so far: \"\n                  + \", \".join(f\"{k} {v[1]}/{v[0]}\" for k, v in stats.items()), flush=True)\n    kept = sum(1 for line in open(a.out) if json.loads(line)[\"answer\"])\n    print(f\"done: {kept} good examples in {a.out}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", "train.py": "\"\"\"Step 2 on Kaggle: teach the small writer (the \"student\") from the teacher's checked answers.\n\n    python train.py --data /kaggle/working/data.jsonl --out /kaggle/working/yb-writer\n\nLoRA on Qwen3-1.7B (Apache-2.0), fp16 on one T4. The prompt is formatted exactly as llama.cpp formats it on the\nwebsite (Qwen3 chat template, thinking off), and only the answer tokens are learned.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\n\nimport torch\nfrom peft import LoraConfig, get_peft_model\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n\ndef load(path, tok, max_len):\n    rows = [json.loads(line) for line in open(path)]\n    rows = [r for r in rows if r[\"answer\"]]\n    rng = random.Random(1)\n    rng.shuffle(rows)\n    n_test = max(20, len(rows) // 20)\n    test, train = rows[:n_test], rows[n_test:]\n    examples = []\n    for r in train:\n        prompt = tok.apply_chat_template([{\"role\": \"system\", \"content\": r[\"system\"]}, {\"role\": \"user\", \"content\": r[\"user\"]}],\n                                         tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        p = tok(prompt, add_special_tokens=False)[\"input_ids\"]\n        a = tok(r[\"answer\"] + \"<|im_end|>\", add_special_tokens=False)[\"input_ids\"]\n        if len(a) > 1400 or len(p) + len(a) > max_len:\n            continue\n        examples.append((p + a, [-100] * len(p) + a))\n    return examples, test\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data\", default=\"/kaggle/working/data.jsonl\")\n    ap.add_argument(\"--out\", default=\"/kaggle/working/yb-writer\")\n    ap.add_argument(\"--student\", default=os.environ.get(\"STUDENT\", \"Qwen/Qwen3-1.7B\"))\n    ap.add_argument(\"--epochs\", type=float, default=2)\n    ap.add_argument(\"--lr\", type=float, default=2e-4)\n    ap.add_argument(\"--accum\", type=int, default=8)\n    ap.add_argument(\"--max_len\", type=int, default=2600)\n    a = ap.parse_args()\n    torch.manual_seed(0)\n    tok = AutoTokenizer.from_pretrained(a.student)\n    examples, test = load(a.data, tok, a.max_len)\n    os.makedirs(a.out, exist_ok=True)\n    with open(os.path.join(a.out, \"test.jsonl\"), \"w\") as f:\n        for r in test:\n            f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n    print(f\"train {len(examples)} examples, held-out test {len(test)}\", flush=True)\n\n    model = AutoModelForCausalLM.from_pretrained(a.student, torch_dtype=torch.float16).cuda()\n    model.gradient_checkpointing_enable()\n    model.enable_input_require_grads()\n    model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type=\"CAUSAL_LM\",\n                                             target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"]))\n    for p in model.parameters():          # adapters train in fp32 for stable fp16 mixed precision\n        if p.requires_grad:\n            p.data = p.data.float()\n    model.print_trainable_parameters()\n    params = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=0.0)\n    steps = math.ceil(len(examples) * a.epochs / a.accum)\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 30) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))\n    scaler = torch.amp.GradScaler(\"cuda\")\n    model.train()\n    rng = random.Random(0)\n    t0, step, seen, run_loss = time.time(), 0, 0, 0.0\n    order = []\n    while len(order) < len(examples) * a.epochs:\n        e = list(range(len(examples)))\n        rng.shuffle(e)\n        order += e\n    order = order[:int(len(examples) * a.epochs)]\n    for i, idx in enumerate(order):\n        ids, labels = examples[idx]\n        x = torch.tensor([ids], device=\"cuda\")\n        y = torch.tensor([labels], device=\"cuda\")\n        with torch.autocast(\"cuda\", dtype=torch.float16):\n            loss = model(input_ids=x, labels=y).loss / a.accum\n        scaler.scale(loss).backward()\n        run_loss += loss.item()\n        seen += 1\n        if seen % a.accum == 0 or i == len(order) - 1:\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0)\n            scaler.step(opt)\n            scaler.update()\n            opt.zero_grad(set_to_none=True)\n            sched.step()\n            step += 1\n            if step % 10 == 0 or step == steps:\n                print(f\"step {step}/{steps} loss {run_loss / 10:.3f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s\", flush=True)\n                run_loss = 0.0\n    model.save_pretrained(os.path.join(a.out, \"adapter\"))\n    merged = model.merge_and_unload()\n    merged = merged.half()\n    merged.save_pretrained(os.path.join(a.out, \"merged\"), safe_serialization=True)\n    tok.save_pretrained(os.path.join(a.out, \"merged\"))\n    print(\"saved\", a.out, f\"{time.time() - t0:.0f}s\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", "evaluate.py": "\"\"\"Step 3 on Kaggle: test the small writer before and after training on held-out examples it never saw.\n\n    python evaluate.py --model /kaggle/working/yb-writer/merged --test /kaggle/working/yb-writer/test.jsonl --out report.md\n\nSame checks as the data builder: valid JSON, right shape and counts, the chosen language's script, honest SEO\nscores. Runs the untrained base model too, so the report shows the real improvement.\n\"\"\"\nimport argparse\nimport collections\nimport json\nimport os\nimport sys\nimport time\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport ybw_data as D  # noqa: E402\n\n\ndef run(name, rows, batch=8):\n    tok = AutoTokenizer.from_pretrained(name, padding_side=\"left\")\n    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()\n    outs = []\n    t0 = time.time()\n    for i in range(0, len(rows), batch):\n        prompts = [tok.apply_chat_template([{\"role\": \"system\", \"content\": r[\"system\"]}, {\"role\": \"user\", \"content\": r[\"user\"]}],\n                                           tokenize=False, add_generation_prompt=True, enable_thinking=False) for r in rows[i:i + batch]]\n        enc = tok(prompts, return_tensors=\"pt\", padding=True, add_special_tokens=False).to(\"cuda\")\n        with torch.no_grad():\n            g = model.generate(**enc, max_new_tokens=1500, do_sample=False, pad_token_id=tok.pad_token_id)\n        outs += tok.batch_decode(g[:, enc[\"input_ids\"].shape[1]:], skip_special_tokens=True)\n    del model\n    torch.cuda.empty_cache()\n    print(name, f\"{time.time() - t0:.0f}s\", flush=True)\n    return outs\n\n\ndef score(rows, outs):\n    by_task, by_lang, fails = collections.defaultdict(lambda: [0, 0]), collections.defaultdict(lambda: [0, 0]), []\n    for r, o in zip(rows, outs):\n        ok, why = D.check(r[\"task\"], dict(r[\"params\"]), D.parse_json(o))\n        for d, k in ((by_task, r[\"task\"]), (by_lang, r[\"lang\"])):\n            d[k][0] += 1\n            d[k][1] += ok\n        if not ok:\n            fails.append((r[\"task\"], r[\"lang\"], why))\n    total = sum(v[1] for v in by_task.values()), sum(v[0] for v in by_task.values())\n    return by_task, by_lang, total, fails\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", default=\"/kaggle/working/yb-writer/merged\")\n    ap.add_argument(\"--base\", default=os.environ.get(\"STUDENT\", \"Qwen/Qwen3-1.7B\"))\n    ap.add_argument(\"--test\", default=\"/kaggle/working/yb-writer/test.jsonl\")\n    ap.add_argument(\"--out\", default=\"/kaggle/working/report.md\")\n    ap.add_argument(\"--limit\", type=int, default=160)\n    a = ap.parse_args()\n    rows = [json.loads(line) for line in open(a.test)][:a.limit]\n    base_outs = run(a.base, rows)\n    new_outs = run(a.model, rows)\n    b, n = score(rows, base_outs), score(rows, new_outs)\n    pct = lambda v: f\"{v[1]}/{v[0]} ({100 * v[1] / max(1, v[0]):.0f}%)\"  # noqa: E731\n    lines = [\"# YB Writer — held-out test\", \"\", f\"{len(rows)} examples the model never saw during training.\", \"\",\n             f\"**Passed all checks:** before training {pct(b[2][::-1])}, after training {pct(n[2][::-1])}\", \"\",\n             \"| Tool task | Before | After |\", \"|---|---|---|\"]\n    for k in sorted(n[0]):\n        lines.append(f\"| {k} | {pct(b[0][k])} | {pct(n[0][k])} |\")\n    lines += [\"\", \"| Language | Before | After |\", \"|---|---|---|\"]\n    for k in sorted(n[1]):\n        lines.append(f\"| {k} | {pct(b[1][k])} | {pct(n[1][k])} |\")\n    lines += [\"\", \"## Why the trained model failed (after)\", \"\"]\n    for k, v in collections.Counter(f\"{t} ({lang}): {w}\" for t, lang, w in n[3]).most_common(25):\n        lines.append(f\"- {k} ×{v}\")\n    lines += [\"\", \"## Sample answers after training (one per task)\", \"\"]\n    shown = set()\n    for r, o in zip(rows, new_outs):\n        if r[\"task\"] in shown:\n            continue\n        shown.add(r[\"task\"])\n        lines += [f\"**{r['task']}** ({r['lang']}) — {r['user'][:160]}\", \"\", \"```\", o[:900], \"```\", \"\"]\n    open(a.out, \"w\").write(\"\\n\".join(lines))\n    print(\"\\n\".join(lines[:40]))\n\n\nif __name__ == \"__main__\":\n    main()\n"}
for name, text in FILES.items():
    open(f'/kaggle/working/ybw/{name}', 'w', encoding='utf-8').write(text)
print(sorted(os.listdir('/kaggle/working/ybw')))


In [ ]:
# Install the tools (~5 minutes). If vLLM fails to install, step 1 uses the slower fallback teacher.
!pip install -q -U vllm 2>&1 | tail -2
!pip install -q -U peft accelerate sentencepiece 2>&1 | tail -2


In [ ]:
# Step 1: the teacher writes and checks the examples (~1-3 hours). Safe to re-run: it resumes.
!cd /kaggle/working/ybw && python gen.py --out /kaggle/working/data.jsonl --scale {SCALE}


In [ ]:
# Step 2: train the small writer (~1-2 hours)
!cd /kaggle/working/ybw && CUDA_VISIBLE_DEVICES=0 python train.py --data /kaggle/working/data.jsonl --out /kaggle/working/yb-writer


In [ ]:
# Step 3: test before vs after on examples it never saw (~20 minutes)
!cd /kaggle/working/ybw && CUDA_VISIBLE_DEVICES=0 python evaluate.py --model /kaggle/working/yb-writer/merged --test /kaggle/working/yb-writer/test.jsonl --out /kaggle/working/report.md


In [ ]:
# Step 4: pack it as one GGUF file for llama.cpp on the ASUS (~15 minutes)
%cd /kaggle/working
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!PYTHONPATH=llama.cpp/gguf-py python llama.cpp/convert_hf_to_gguf.py yb-writer/merged --outtype f16 --outfile yb-writer-f16.gguf 2>&1 | tail -2
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j 4 2>&1 | tail -1
!llama.cpp/build/bin/llama-quantize yb-writer-f16.gguf yb-writer-q4_k_m.gguf Q4_K_M 2>&1 | tail -2


In [ ]:
# Step 5: tidy up and show the results
import hashlib, shutil, os
os.chdir('/kaggle/working')
for p in ['yb-writer-f16.gguf', 'llama.cpp', 'yb-writer/merged']:
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else (os.remove(p) if os.path.exists(p) else None)
h = hashlib.sha256()
with open('yb-writer-q4_k_m.gguf', 'rb') as f:
    for b in iter(lambda: f.read(1 << 20), b''):
        h.update(b)
open('yb-writer-q4_k_m.gguf.sha256', 'w').write(h.hexdigest() + '  yb-writer-q4_k_m.gguf\n')
print('GGUF size: %.2f GB  sha256: %s' % (os.path.getsize('yb-writer-q4_k_m.gguf') / 1e9, h.hexdigest()))
print(open('report.md').read()[:3000])
